# CranioVision - nnU-Net-Style DynUNet Clinical Demo

This notebook runs the **complete CranioVision pipeline** on one brain MRI case:

1. **Segmentation** — nnU-Net-style DynUNet predicts tumor regions
2. **Quantification** — Per-region tumor volumes in cm³
3. **Uncertainty** — MC Dropout confidence estimation
4. **Explainability** — Grad-CAM shows what the model is looking at
5. **Clinical report** — Unified summary in doctor-friendly format

This is the **demo notebook** — what you show a supervisor, a reviewer, or a potential clinical partner.

**Runtime:** ~10 minutes on GTX 1650 4GB (includes MC Dropout with 5 samples)

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import torch

from src.cranovision.config import MODELS_DIR, OUTPUTS_DIR, CLASS_NAMES
from src.cranovision.data import get_splits
from src.cranovision.inference import (
    load_model,
    predict_case,
    make_inferer,
    compute_region_volumes,
    mc_dropout_predict,
    summarize_confidence,
    compute_grad_cam,
)
from src.cranovision.training.metrics import (
    compute_case_dice,
    compute_brats_region_dice,
)

print('✅ Imports ok')

In [ ]:
# ── Configuration ──
CASE_IDX     = 0       # which test case to analyze (0-29)
N_MC_SAMPLES = 5       # MC Dropout stochastic passes (5 = fast, 20 = publication)
GRADCAM_PATCH_SIZE = (64, 64, 64)  # Grad-CAM crop only; smaller saves VRAM
MODEL_NAME   = 'nnunet'
MODEL_DISPLAY = 'nnU-Net-style DynUNet'
CKPT_FILE     = 'nnunet_best.pth'

## 2. Load Model + Pick Case

In [ ]:
model = load_model(MODEL_NAME, MODELS_DIR / CKPT_FILE)
inferer = make_inferer()

_, _, test_cases = get_splits()
case = test_cases[CASE_IDX]
CASE_ID = case['case_id']
print(f'\nTarget case: {CASE_ID}')

## 3. Stage 1 — Segmentation + Volume Quantification

In [ ]:
print('Running segmentation...')
seg_result = predict_case(model, case, inferer=inferer)

pred_volumes = compute_region_volumes(seg_result['pred'])
true_volumes = compute_region_volumes(seg_result['label'].squeeze(0))
dice_per_class = compute_case_dice(seg_result['pred'], seg_result['label'].squeeze(0))
brats_dice = compute_brats_region_dice(seg_result['pred'], seg_result['label'].squeeze(0))

print(f'\n✓ Segmentation complete')
print(f'  Total tumor volume: {pred_volumes["Total tumor"]:.2f} cm³')
print(f'  Mean Dice vs GT   : {np.mean(dice_per_class):.4f}')

## 4. Stage 2 — Uncertainty Quantification (MC Dropout)

In [ ]:
print(f'Running MC Dropout ({N_MC_SAMPLES} samples)... ~{N_MC_SAMPLES} min')
mc_result = mc_dropout_predict(
    model=model,
    case_dict=case,
    n_samples=N_MC_SAMPLES,
    inferer=inferer,
    verbose=True,
)
confidence = summarize_confidence(mc_result, uncertain_threshold=0.15)

print(f'\n✓ Uncertainty quantified')
print(f'  Mean confidence  : {confidence["mean_confidence"]:.4f}')
print(f'  Uncertain voxels : {confidence["uncertain_voxel_count"]:,} ({confidence["uncertain_fraction"]*100:.2f}%)')

## 5. Stage 3 — Explainability (Grad-CAM)

In [ ]:
print(f'Running Grad-CAM on patch {GRADCAM_PATCH_SIZE}... ~1 min')
cam_result = compute_grad_cam(
    model=model,
    case_dict=case,
    model_name=MODEL_NAME,
    target_classes=(1, 2, 3),
    use_predicted_mask=True,
    patch_size=GRADCAM_PATCH_SIZE,
    verbose=True,
)
print(f'\n✓ Grad-CAM complete')
print(f'  Heatmaps generated for {len(cam_result["heatmaps"])} classes')

## 6. Clinical Report

In [ ]:
# Unified clinical summary
print('╔' + '═' * 70 + '╗')
print(f'║  CRANIOVISION CLINICAL REPORT{" " * 40}║')
print(f'║  Case: {CASE_ID:<62}║')
print('╚' + '═' * 70 + '╝')

print('\n▸ SEGMENTATION')
print(f'  Model: {MODEL_DISPLAY}')
print()
print(f'  {"Region":<22}{"Volume (cm³)":>15}{"GT volume":>15}{"Dice":>10}')
print(f'  {"-" * 62}')
for i, name in enumerate(CLASS_NAMES[1:]):
    print(f'  {name:<22}{pred_volumes[name]:>13.2f}  '
          f'{true_volumes[name]:>13.2f}  '
          f'{dice_per_class[i]:>8.4f}')
print(f'  {"-" * 62}')
print(f'  {"Total":<22}{pred_volumes["Total tumor"]:>13.2f}  '
      f'{true_volumes["Total tumor"]:>13.2f}')

print('\n▸ BraTS STANDARD REGIONS')
print(f'  Whole Tumor (WT)     : {brats_dice["WT"]:.4f}')
print(f'  Tumor Core  (TC)     : {brats_dice["TC"]:.4f}')
print(f'  Enhancing   (ET)     : {brats_dice["ET"]:.4f}')

print('\n▸ CONFIDENCE ASSESSMENT (MC Dropout, {} samples)'.format(N_MC_SAMPLES))
print(f'  Overall mean confidence  : {confidence["mean_confidence"]:.4f}')
print(f'  Uncertain voxel fraction : {confidence["uncertain_fraction"]*100:.2f}%')
for name, stats in confidence['per_class'].items():
    flag = ' ⚠ REVIEW' if stats['mean_confidence'] < 0.85 else ' ✓'
    print(f'  {name:<22}: {stats["mean_confidence"]:.4f}{flag}')

# Auto-flag
if confidence['mean_confidence'] >= 0.90:
    verdict = '✓ HIGH CONFIDENCE — AI prediction reliable'
elif confidence['mean_confidence'] >= 0.75:
    verdict = '~ MODERATE CONFIDENCE — quick radiologist review suggested'
else:
    verdict = '⚠ LOW CONFIDENCE — MANDATORY radiologist review'

print('\n▸ VERDICT')
print(f'  {verdict}')
print()
print('─' * 72)
print(f'  Generated by CranioVision | Visualization outputs saved below')
print('─' * 72)

## 7. Unified Visual Report — 12 panels, one figure

The complete clinical view in one glance.

In [ ]:
# Custom colormaps
unc_cmap = LinearSegmentedColormap.from_list(
    'uncertainty', ['#0a0a0a', '#3a1a00', '#cc5500', '#ffaa33', '#ffffff'])
gradcam_cmap = LinearSegmentedColormap.from_list(
    'gradcam', ['#0a0a0a', '#331100', '#aa3300', '#ff6622', '#ffdd44', '#ffffff'])

def overlay_seg(mri, seg):
    mri_n = (mri - mri.min()) / (mri.max() - mri.min() + 1e-8)
    rgb = np.stack([mri_n, mri_n, mri_n], axis=-1)
    colors = {1: [1.0, 0.85, 0], 2: [1.0, 0.2, 0.1], 3: [0.2, 0.5, 1.0]}
    for lbl, color in colors.items():
        m = seg == lbl
        if m.any():
            rgb[m] = 0.5 * rgb[m] + 0.5 * np.array(color)
    return np.clip(rgb, 0, 1)

def overlay_heatmap(mri, heatmap, alpha=0.55):
    mri_n = (mri - mri.min()) / (mri.max() - mri.min() + 1e-8)
    rgb = np.stack([mri_n, mri_n, mri_n], axis=-1)
    hm_n = heatmap / (heatmap.max() + 1e-8)
    colored = gradcam_cmap(hm_n)[..., :3]
    mask = hm_n > 0.05
    for c in range(3):
        rgb[..., c] = np.where(mask,
                                (1 - alpha * hm_n) * rgb[..., c] + alpha * hm_n * colored[..., c],
                                rgb[..., c])
    return np.clip(rgb, 0, 1)

# Find optimal slice
pred = seg_result['pred'].numpy()
tumor_per_slice = (pred > 0).sum(axis=(0, 1))
TARGET_SLICE = int(tumor_per_slice.argmax())

# Extract slice data
img_slice = seg_result['image'][1, :, :, TARGET_SLICE].numpy().T    # T1c
gt_slice  = seg_result['label'].squeeze(0)[:, :, TARGET_SLICE].numpy().T
pr_slice  = pred[:, :, TARGET_SLICE].T
unc_slice = mc_result['uncertainty'].numpy()[:, :, TARGET_SLICE].T
img_n = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-8)

cam_slices = {cls: cam_result['heatmaps'][cls][:, :, TARGET_SLICE].numpy().T
              for cls in (1, 2, 3)}

# 4 rows × 3 cols = 12 panels
fig = plt.figure(figsize=(16, 20))
fig.patch.set_facecolor('#0f0f0f')
gs = fig.add_gridspec(4, 3, hspace=0.18, wspace=0.05)

panels = [
    ('T1c (input MRI)',       img_n,                            'gray',    None),
    ('Ground truth',          overlay_seg(img_slice, gt_slice), None,      'rgb'),
    ('Prediction',            overlay_seg(img_slice, pr_slice), None,      'rgb'),

    ('Uncertainty heatmap',   unc_slice,                        unc_cmap,  (0, 0.3)),
    ('Seg + uncertainty',     overlay_seg(img_slice, pr_slice), None,      'rgb'),
    ('T1c + uncertainty',     overlay_heatmap(img_slice, unc_slice / 0.3), None, 'rgb'),

    ('Grad-CAM: Edema',       overlay_heatmap(img_slice, cam_slices[1]),   None, 'rgb'),
    ('Grad-CAM: Enhancing',   overlay_heatmap(img_slice, cam_slices[2]),   None, 'rgb'),
    ('Grad-CAM: Necrotic',    overlay_heatmap(img_slice, cam_slices[3]),   None, 'rgb'),

    ('Pure edema heatmap',    cam_slices[1],     gradcam_cmap, (0, 1)),
    ('Pure enhancing heatmap',cam_slices[2],     gradcam_cmap, (0, 1)),
    ('Pure necrotic heatmap', cam_slices[3],     gradcam_cmap, (0, 1)),
]

for i, (title, data, cmap, spec) in enumerate(panels):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    if spec == 'rgb':
        ax.imshow(data, origin='lower')
    elif spec is None:
        ax.imshow(data, cmap=cmap, origin='lower')
    else:
        ax.imshow(data, cmap=cmap, origin='lower', vmin=spec[0], vmax=spec[1])

    ax.set_title(title, color='white', fontsize=12)
    ax.tick_params(colors='white', labelsize=0, length=0)
    ax.set_facecolor('#0f0f0f')
    for s in ax.spines.values():
        s.set_edgecolor('#333')

# Row labels on the left
row_labels = ['Input + GT/Prediction', 'Uncertainty', 'Grad-CAM overlays', 'Pure Grad-CAM']
for row_idx, label in enumerate(row_labels):
    fig.text(0.03, 0.87 - row_idx * 0.235, label, color='white',
             rotation=90, fontsize=13, va='center', fontweight='bold')

# Main title
plt.suptitle(
    f'CranioVision — Complete Clinical Pipeline\n'
    f'{CASE_ID}  |  slice z={TARGET_SLICE}  |  '
    f'Total tumor {pred_volumes["Total tumor"]:.1f} cm³  |  '
    f'Mean Dice {np.mean(dice_per_class):.3f}  |  '
    f'Confidence {confidence["mean_confidence"]:.3f}',
    color='white', fontsize=15, y=0.995)

# Legend
legend = [
    mpatches.Patch(color='#FFD900', label='Edema'),
    mpatches.Patch(color='#FF3322', label='Enhancing'),
    mpatches.Patch(color='#3377FF', label='Necrotic'),
    mpatches.Patch(color='#ff8c00', label='Uncertain / attention'),
]
fig.legend(handles=legend, loc='lower center', ncol=4,
           facecolor='#1a1a1a', labelcolor='white', fontsize=11,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

save_path = OUTPUTS_DIR / f'nnunet_demo_{CASE_ID}.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f'\n✅ Saved: {save_path}')

## 8. Save JSON clinical summary (machine-readable)

In [ ]:
import json

report = {
    'case_id': CASE_ID,
    'model': MODEL_NAME,
    'model_display': MODEL_DISPLAY,
    'checkpoint': CKPT_FILE,
    'segmentation': {
        'volumes_cm3': pred_volumes,
        'ground_truth_volumes_cm3': true_volumes,
        'dice_per_class': {name: float(d) for name, d in zip(CLASS_NAMES[1:], dice_per_class)},
        'brats_regions_dice': {k: float(v) for k, v in brats_dice.items()},
    },
    'uncertainty': {
        'n_samples': N_MC_SAMPLES,
        'mean_confidence': confidence['mean_confidence'],
        'uncertain_voxel_count': confidence['uncertain_voxel_count'],
        'uncertain_fraction': confidence['uncertain_fraction'],
        'per_class': confidence['per_class'],
    },
    'explainability': {
        'method': 'Grad-CAM 3D (patch-based)',
        'target_layer': cam_result['target_layer'],
        'patch_offset': list(cam_result['patch_offset']),
    },
    'verdict': verdict,
}

report_path = OUTPUTS_DIR / f'nnunet_demo_{CASE_ID}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f'✅ JSON report saved: {report_path}')
print(f'\nAll outputs from this demo:')
print(f'  1. {save_path}')
print(f'  2. {report_path}')